# Week 3 — Occupation Audit
**Goal:** Before running any Nemotron experiments, manually audit 30 randomly sampled occupations
from the dataset and verify the sample is balanced across yes / no / maybe college-required categories.

**Instructions:**
1. Run Cell 1 — see the 30 occupations and histogram
2. Fill in `occupation_audit` in Cell 2 manually
3. Run Cell 3 — check balance and decide whether to proceed

> **Do not run Nemotron experiments until Cell 3 confirms the sample is balanced.**

## 1. Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import os

sns.set_theme(style="whitegrid")

SAMPLE_PATH = "../data/personas_sample_500.csv"
RESULTS_DIR = "../results"
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Imports OK")

## 2. Load Sample (No Cleaning)
Load the raw sample as-is — do not filter or fix any labels yet.

In [ ]:
# Load raw sample — NO cleaning per advisor instruction
sample_raw = pd.read_csv(SAMPLE_PATH)

print(f"Sample size: {len(sample_raw)} rows")
print(f"Unique occupations in sample: {sample_raw['occupation'].nunique()}")
print(f"\nLabel balance:")
print(sample_raw["label_name"].value_counts())
print()
sample_raw.head(3)

## 3. Sample 30 Random Occupations + Histogram

In [ ]:
# Sample 30 random unique occupations from the dataset
all_occupations = (
    sample_raw["occupation"]
    .str.replace("_", " ")
    .str.strip()
    .unique()
    .tolist()
)

random.seed(42)
sampled_30 = sorted(random.sample(all_occupations, 30))

print("=== 30 randomly sampled occupations ===")
for i, occ in enumerate(sampled_30, 1):
    print(f"  {i:2d}. {occ}")

# Count how many times each appears in the full sample
occ_counts = (
    sample_raw["occupation"]
    .str.replace("_", " ")
    .str.strip()
    .value_counts()
)

sampled_counts_df = pd.DataFrame(
    [(occ, occ_counts.get(occ, 0)) for occ in sampled_30],
    columns=["occupation", "count"]
).sort_values("count", ascending=False).reset_index(drop=True)

# Histogram
fig, ax = plt.subplots(figsize=(12, 8))
sns.barplot(data=sampled_counts_df, x="count", y="occupation",
            color="#378ADD", ax=ax)
ax.set_title("30 Sampled Occupations: Frequency in 500-row Sample",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Count in sample")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/occupation_sample_audit.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/occupation_sample_audit.png")


## 4. Manual Audit

Look at the 30 occupations printed above and fill in the dictionary below.

**Categories:**
- `"yes"` : this occupation clearly requires a college degree (e.g. surgeon, software engineer)
- `"no"` : this occupation clearly does not require a college degree (e.g. cashier, janitor)
- `"maybe"` : this occupation is ambiguous, could go either way (e.g. manager, supervisor)

> Fill in all 30 occupations, then run Cell 5 to check balance.

In [ ]:
occupation_audit = {
    "advertising or promotions manager":                                          "yes",
    "animal trainer":                                                             "no",
    "barber":                                                                     "no",
    "budget analyst":                                                             "yes",
    "chief executive":                                                            "yes",
    "chiropractor":                                                               "yes",
    "compliance officer":                                                         "maybe",
    "construction equipment operator":                                            "no",
    "credit counselor or loan officer":                                           "maybe",
    "customer service representative":                                            "no",
    "dental or ophthalmic laboratory technician or medical appliance technician": "maybe",
    "designer":                                                                   "maybe",
    "fast food or counter worker":                                                "no",
    "first line supervisor of housekeeping or janitorial worker":                 "no",
    "industrial truck or tractor operator":                                       "no",
    "information security analyst":                                               "yes",
    "inspector tester sorter sampler or weigher":                                 "no",
    "laundry or dry cleaning worker":                                             "no",
    "mechanical engineer":                                                        "yes",
    "office clerk general":                                                       "no",
    "pharmacist":                                                                 "yes",
    "physical therapist":                                                         "yes",
    "preschool or kindergarten teacher":                                          "yes",
    "reservation or transportation ticket agent or travel clerk":                 "no",
    "social or human service assistant":                                          "maybe",
    "surgical technologist":                                                      "yes",
    "teacher or instructor":                                                      "yes",
    "training or development specialist":                                         "maybe",
    "weigher measurer checker or sampler recordkeeping":                          "no",
    "wholesale or retail buyer":                                                  "maybe",
}

print(f"Filled in: {len(occupation_audit)} / 30 occupations")
if len(occupation_audit) < 30:
    print("Fill in all 30 above, then run Cell 5.")
else:
    print("All 30 filled in — run Cell 5 to check balance.")

## 5. Balance Check — Proceed or Not?

In [ ]:
if not occupation_audit:
    print("occupation_audit is empty — fill it in Cell 4 first.")
elif len(occupation_audit) < 30:
    print(f"Only {len(occupation_audit)}/30 filled in — complete Cell 4 first.")
else:
    audit_df = pd.DataFrame(
        [(occ, verdict) for occ, verdict in occupation_audit.items()],
        columns=["occupation", "verdict"]
    )

    # ── Results by category ───────────────────────────────────────────
    print("=== OCCUPATION AUDIT RESULTS ===")
    for verdict in ["yes", "no", "maybe"]:
        group = audit_df[audit_df["verdict"] == verdict]["occupation"].tolist()
        print(f"\n{verdict.upper()} — {len(group)} occupations:")
        for occ in group:
            print(f"  - {occ}")

    # ── Balance check ─────────────────────────────────────────────────
    counts    = audit_df["verdict"].value_counts()
    yes_pct   = counts.get("yes",   0) / 30 * 100
    no_pct    = counts.get("no",    0) / 30 * 100
    maybe_pct = counts.get("maybe", 0) / 30 * 100

    print(f"\n=== BALANCE CHECK ===")
    print(f"  Yes   (college):    {counts.get('yes',   0):2d} / 30  ({yes_pct:.0f}%)")
    print(f"  No    (not college):{counts.get('no',    0):2d} / 30  ({no_pct:.0f}%)")
    print(f"  Maybe (ambiguous):  {counts.get('maybe', 0):2d} / 30  ({maybe_pct:.0f}%)")

    # ── Visualization ─────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Bar chart of yes/no/maybe counts
    colors = ["#1D9E75", "#378ADD", "#EF9F27"]
    vc = audit_df["verdict"].value_counts().reindex(["yes", "no", "maybe"], fill_value=0)
    axes[0].bar(vc.index, vc.values, color=colors)
    axes[0].set_title("Audit Distribution — Yes / No / Maybe")
    axes[0].set_ylabel("Count")
    for i, (label, val) in enumerate(vc.items()):
        axes[0].text(i, val + 0.2, str(val), ha="center", fontsize=12, fontweight="bold")

    # Horizontal bar chart colored by verdict
    color_map  = {"yes": "#1D9E75", "no": "#378ADD", "maybe": "#EF9F27"}
    audit_plot = audit_df.sort_values("verdict")
    bar_colors = [color_map[v] for v in audit_plot["verdict"]]
    axes[1].barh(audit_plot["occupation"], [1]*30, color=bar_colors)
    axes[1].set_title("Each Occupation Colored by Verdict")
    axes[1].set_xlabel("")
    axes[1].tick_params(axis="y", labelsize=8)
    axes[1].set_xticks([])

    # Legend
    from matplotlib.patches import Patch
    legend = [Patch(color="#1D9E75", label="Yes"),
              Patch(color="#378ADD", label="No"),
              Patch(color="#EF9F27", label="Maybe")]
    axes[1].legend(handles=legend, loc="lower right")

    plt.suptitle("Occupation Verification (Balance Check)", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f"{RESULTS_DIR}/occupation_audit_balance.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: results/occupation_audit_balance.png")

    # ── Decision ──────────────────────────────────────────────────────
    print("\n=== DECISION ===")
    if yes_pct >= 25 and no_pct >= 25 and maybe_pct >= 15:
        print("✓ Sample is balanced — safe to proceed with Nemotron experiments.")
        print("  Next step: add the experiment cells and run Week 3.")
    else:
        print("⚠ Sample may not be balanced enough.")
        if yes_pct < 25:
            print(f"  - Not enough YES occupations ({yes_pct:.0f}% — need at least 25%)")
        if no_pct < 25:
            print(f"  - Not enough NO occupations ({no_pct:.0f}% — need at least 25%)")
        if maybe_pct < 15:
            print(f"  - Not enough MAYBE occupations ({maybe_pct:.0f}% — need at least 15%)")
        print("  Discuss with your advisor before proceeding.")

    # Save audit results
    audit_df.to_csv(f"{RESULTS_DIR}/occupation_audit_results.csv", index=False)
    print("\nSaved: results/occupation_audit_results.csv")

## 6. Map Audit Verdicts to Sample

Tag every row in the 500-row sample whose occupation matches one of the 30 audited ones.
Rows with unaudited occupations are tagged as `unaudited`.

In [ ]:
# Normalize occupation names to match audit keys
sample_raw["occ_normalized"] = (
    sample_raw["occupation"]
    .str.replace("_", " ")
    .str.strip()
    .str.lower()
)

audit_normalized = {k.lower(): v for k, v in occupation_audit.items()}

sample_raw["audit_verdict"] = sample_raw["occ_normalized"].map(audit_normalized)

tagged   = sample_raw[sample_raw["audit_verdict"].notna()]
untagged = sample_raw[sample_raw["audit_verdict"].isna()]

print(f"Total rows:    {len(sample_raw)}")
print(f"Tagged rows:   {len(tagged)} ({len(tagged)/len(sample_raw)*100:.1f}%)")
print(f"Untagged rows: {len(untagged)} ({len(untagged)/len(sample_raw)*100:.1f}%)")
print()
print("Tagged rows by verdict:")
print(tagged["audit_verdict"].value_counts())
print()
print("Tagged rows by verdict and true label:")
print(tagged.groupby(["audit_verdict", "label_name"]).size().unstack(fill_value=0))


## 7. True Histogram — Occupation Frequency Distribution

Shows how many occupations appear once, twice, three times, etc. in the sample.

In [ ]:
occ_all_counts = (
    sample_raw["occupation"]
    .str.replace("_", " ")
    .str.strip()
    .value_counts()
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(occ_all_counts.values, bins=20, color="#378ADD", edgecolor="white")
ax.set_title("Histogram — Occupation Frequency Distribution in 500-row Sample",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Number of times an occupation appears in sample")
ax.set_ylabel("Number of occupations with that frequency")
plt.tight_layout()
plt.savefig("../results/occupation_frequency_histogram.png", dpi=150)
plt.show()
print("Saved: results/occupation_frequency_histogram.png")
print(f"\nMost common: '{occ_all_counts.index[0]}' appears {occ_all_counts.iloc[0]} times")
print(f"Appearing only once: {(occ_all_counts == 1).sum()} occupations")
print(f"Appearing 2-5 times: {((occ_all_counts >= 2) & (occ_all_counts <= 5)).sum()} occupations")
print(f"Appearing 6+ times:  {(occ_all_counts >= 6).sum()} occupations")


## 8. Helper Functions & Few-shot Examples

In [ ]:
import requests
import time
import re
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder, StandardScaler

OLLAMA_URL = "http://localhost:11434/v1/chat/completions"
MODEL_NAME = "nemotron-3-nano:4b"

SYSTEM_PROMPT = """You are an education level classifier. Given a person's demographic information,
predict whether they are college-educated (have a bachelor's degree or higher) or not.
Think step by step, then respond with ONLY one of these two labels on the final line: college or not_college."""


def serialize_row(row):
    return (
        f"A {int(row['age'])}-year-old {str(row['sex']).lower().strip()}, "
        f"{str(row['marital_status']).replace('_', ' ').strip()}, "
        f"working as a {str(row['occupation']).replace('_', ' ').strip()}. "
        f"Located in {str(row['state']).strip()}."
    )


def build_zero_shot_prompt(row):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
            f"Classify this person's education level:\n\n"
            f"{serialize_row(row)}\n\n"
            f"Answer with college or not_college only."
        )}
    ]


def build_few_shot_prompt(row, examples):
    example_text = ""
    for ex in examples:
        example_text += f"Person: {serialize_row(ex)}\nLabel: {ex['label_name']}\n\n"
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
            f"Here are some labeled examples:\n\n{example_text}"
            f"Now classify this person:\n\n"
            f"{serialize_row(row)}\n\n"
            f"Answer with college or not_college only."
        )}
    ]


def parse_response(content):
    think_match = re.search(r"<think>(.*?)</think>", content, re.DOTALL)
    trace = think_match.group(1).strip() if think_match else ""
    label_match = re.search(r"\b(not_college|college)\b", content, re.IGNORECASE)
    label = label_match.group(1).lower() if label_match else "UNKNOWN"
    return label, trace


def classify_row(messages, timeout=120):
    payload = {
        "model": MODEL_NAME,
        "messages": messages,
        "max_tokens": 1024,
        "temperature": 0.1,
        "stream": False,
        "options": {"num_ctx": 4096},
    }
    start = time.time()
    try:
        response = requests.post(OLLAMA_URL, json=payload, timeout=timeout).json()
        elapsed  = time.time() - start
        content  = response["choices"][0]["message"]["content"]
        tokens   = response.get("usage", {}).get("completion_tokens", 0)
        label, trace = parse_response(content)
    except Exception as e:
        elapsed = time.time() - start
        label, trace, tokens, content = "ERROR", str(e), 0, str(e)
    return {"label": label, "trace": trace, "raw": content,
            "time_ms": round(elapsed * 1000), "tokens": tokens}


# Few-shot examples — must come from audited rows only
# This ensures every example has a manually verified occupation label
audited_rows = sample_raw[sample_raw["audit_verdict"].notna()]

fs_college = audited_rows[audited_rows["label_name"] == "college"].sample(3, random_state=1)
fs_not     = audited_rows[audited_rows["label_name"] == "not_college"].sample(2, random_state=1)
few_shot_examples = pd.concat([fs_college, fs_not]).sample(frac=1, random_state=1).to_dict("records")

print(f"Audited rows available: {len(audited_rows)}")
print(f"  College: {len(audited_rows[audited_rows['label_name'] == 'college'])}")
print(f"  Not college: {len(audited_rows[audited_rows['label_name'] == 'not_college'])}")

print("Few-shot examples:")
for ex in few_shot_examples:
    verdict = audit_normalized.get(ex["occupation"].replace("_", " ").strip().lower(), "unaudited")
    print(f"  {serialize_row(ex)} → {ex['label_name']} [{verdict}]")
print(f"\nModel: {MODEL_NAME}")
print("Ready to run experiments.")


## 9. Zero-shot Experiment (500 rows)

> **Expected time:** ~30 minutes.

In [ ]:
print(f"Starting zero-shot on {len(sample_raw)} rows...")
print(f"Estimated time: ~{len(sample_raw) * 4 / 60:.0f} minutes\n")

zs_results = []
t_start = time.time()

for i, row in sample_raw.iterrows():
    messages = build_zero_shot_prompt(row.to_dict())
    result   = classify_row(messages)
    zs_results.append({
        "row_id":        i,
        "input":         serialize_row(row.to_dict()),
        "occupation":    row["occupation"].replace("_", " ").strip(),
        "audit_verdict": row.get("audit_verdict", "unaudited"),
        "true_label":    row["label_name"],
        "pred_label":    result["label"],
        "correct":       result["label"] == row["label_name"],
        "time_ms":       result["time_ms"],
        "tokens":        result["tokens"],
        "trace":         result["trace"],
        "raw":           result["raw"],
    })
    if (i + 1) % 50 == 0:
        elapsed   = time.time() - t_start
        done      = i + 1
        remaining = (elapsed / done) * (len(sample_raw) - done)
        print(f"  Row {done:3d}/{len(sample_raw)} | Elapsed: {elapsed/60:.1f}min | Remaining: ~{remaining/60:.1f}min")

zs_df = pd.DataFrame(zs_results)
os.makedirs(RESULTS_DIR, exist_ok=True)
zs_df.to_csv(f"{RESULTS_DIR}/week3_zeroshot_raw.csv", index=False)
print(f"\nDone! Total time: {(time.time()-t_start)/60:.1f} minutes")
print(f"Saved: results/week3_zeroshot_raw.csv")


## 10. Zero-shot Results

In [ ]:
label_map = {"college": 1, "not_college": 0}

def evaluate_llm_results(results_df, run_name):
    valid = results_df[results_df["pred_label"].isin(["college", "not_college"])].copy()
    unknown_count = len(results_df) - len(valid)
    y_true = valid["true_label"].map(label_map)
    y_pred = valid["pred_label"].map(label_map)
    acc  = accuracy_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred, average="macro")
    auc  = roc_auc_score(y_true, y_pred)
    avg_ms = results_df["time_ms"].mean()
    print(f"\n{'='*50}\n  {run_name}\n{'='*50}")
    print(f"  Rows evaluated: {len(valid)} / {len(results_df)}")
    print(f"  Unknown:        {unknown_count}")
    print(f"  Accuracy:       {acc:.4f}")
    print(f"  Macro F1:       {f1:.4f}")
    print(f"  AUC-ROC:        {auc:.4f}")
    print(f"  Avg time/row:   {avg_ms:.0f}ms")
    print()
    print(classification_report(y_true, y_pred, target_names=["not_college", "college"]))
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(4, 3))
    ConfusionMatrixDisplay(cm, display_labels=["not_college", "college"]).plot(ax=ax, colorbar=False)
    ax.set_title(run_name)
    plt.tight_layout()
    fname = f"{RESULTS_DIR}/cm_{run_name.replace(' ', '_').lower()}.png"
    plt.savefig(fname, dpi=150)
    plt.show()
    return {"model": run_name, "accuracy": round(acc, 4), "macro_f1": round(f1, 4),
            "auc_roc": round(auc, 4), "ms_per_row": round(avg_ms, 1),
            "unknown": unknown_count, "n_samples": len(valid),
            "week": 3, "device": "local-rtx4060",
            "dataset": "nvidia/Nemotron-Personas-USA"}

all_results = []
zs_metrics = evaluate_llm_results(zs_df, "Nano 4B zero-shot")
all_results.append(zs_metrics)


## 11. Few-shot Experiment (500 rows)

> **Expected time:** ~60 minutes.

In [ ]:
print(f"Starting few-shot on {len(sample_raw)} rows...")

fs_results = []
t_start = time.time()

for i, row in sample_raw.iterrows():
    messages = build_few_shot_prompt(row.to_dict(), few_shot_examples)
    result   = classify_row(messages)
    fs_results.append({
        "row_id":        i,
        "input":         serialize_row(row.to_dict()),
        "occupation":    row["occupation"].replace("_", " ").strip(),
        "audit_verdict": row.get("audit_verdict", "unaudited"),
        "true_label":    row["label_name"],
        "pred_label":    result["label"],
        "correct":       result["label"] == row["label_name"],
        "time_ms":       result["time_ms"],
        "tokens":        result["tokens"],
        "trace":         result["trace"],
        "raw":           result["raw"],
    })
    if (i + 1) % 50 == 0:
        elapsed   = time.time() - t_start
        done      = i + 1
        remaining = (elapsed / done) * (len(sample_raw) - done)
        print(f"  Row {done:3d}/{len(sample_raw)} | Elapsed: {elapsed/60:.1f}min | Remaining: ~{remaining/60:.1f}min")

fs_df = pd.DataFrame(fs_results)
fs_df.to_csv(f"{RESULTS_DIR}/week3_fewshot_raw.csv", index=False)
print(f"\nDone! Total time: {(time.time()-t_start)/60:.1f} minutes")
print(f"Saved: results/week3_fewshot_raw.csv")


## 12. Few-shot Results

In [ ]:
fs_metrics = evaluate_llm_results(fs_df, "Nano 4B few-shot")
all_results.append(fs_metrics)


## 13. Accuracy by Occupation Difficulty (Yes / No / Maybe)

The key analysis — accuracy for each audit category.
Shows which types of personas Nemotron identifies well vs poorly.

In [ ]:
print("=== ACCURACY BY OCCUPATION DIFFICULTY ===")
verdict_labels = ["yes", "no", "maybe"]
zs_accs, fs_accs, sizes = [], [], []

for verdict in verdict_labels:
    zs_sub   = zs_df[zs_df["audit_verdict"] == verdict]
    fs_sub   = fs_df[fs_df["audit_verdict"] == verdict]
    zs_valid = zs_sub[zs_sub["pred_label"].isin(["college", "not_college"])]
    fs_valid = fs_sub[fs_sub["pred_label"].isin(["college", "not_college"])]
    zs_acc   = (zs_valid["pred_label"] == zs_valid["true_label"]).mean() if len(zs_valid) > 0 else 0
    fs_acc   = (fs_valid["pred_label"] == fs_valid["true_label"]).mean() if len(fs_valid) > 0 else 0
    zs_accs.append(zs_acc)
    fs_accs.append(fs_acc)
    sizes.append(len(zs_sub))
    top_occs = zs_sub["occupation"].value_counts().head(3).index.tolist()
    print(f"\n{verdict.upper()} ({len(zs_sub)} rows):")
    print(f"  Zero-shot: {zs_acc:.1%}")
    print(f"  Few-shot:  {fs_acc:.1%}")
    print(f"  Top occs:  {top_occs}")

# Visualization
x = range(3)
width = 0.35
fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar([i - width/2 for i in x], zs_accs, width,
               label="Zero-shot", color="#378ADD", alpha=0.85)
bars2 = ax.bar([i + width/2 for i in x], fs_accs, width,
               label="Few-shot",  color="#1D9E75", alpha=0.85)
ax.set_xticks(list(x))
ax.set_xticklabels([f"{v.upper()}\n({s} rows)" for v, s in zip(verdict_labels, sizes)])
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1)
ax.set_title("Nano 4B Accuracy by Occupation Difficulty",
             fontsize=12, fontweight="bold")
ax.legend()
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{bar.get_height():.1%}", ha="center", fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{bar.get_height():.1%}", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/week3_accuracy_by_difficulty.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/week3_accuracy_by_difficulty.png")


## 14. K-Means Cluster Analysis

K-Means with dominant audit verdict per cluster to understand
what each cluster represents.

In [ ]:
cluster_features = ["age", "sex", "marital_status", "occupation", "state"]
X_cluster = sample_raw[cluster_features].copy()

le_dict = {}
for col in ["sex", "marital_status", "occupation", "state"]:
    le = LabelEncoder()
    X_cluster[col] = le.fit_transform(X_cluster[col].astype(str))
    le_dict[col] = le

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

K  = 4
km = KMeans(n_clusters=K, random_state=42, n_init=10)
cluster_labels_arr = km.fit_predict(X_scaled)

zs_df["cluster"] = cluster_labels_arr
fs_df["cluster"] = cluster_labels_arr

cluster_stats = []
for c in range(K):
    zs_rows  = zs_df[zs_df["cluster"] == c]
    fs_rows  = fs_df[fs_df["cluster"] == c]
    zs_valid = zs_rows[zs_rows["pred_label"].isin(["college", "not_college"])]
    fs_valid = fs_rows[fs_rows["pred_label"].isin(["college", "not_college"])]
    zs_acc   = (zs_valid["pred_label"] == zs_valid["true_label"]).mean()
    fs_acc   = (fs_valid["pred_label"] == fs_valid["true_label"]).mean()
    size     = len(zs_rows)
    college_rate   = (zs_rows["true_label"] == "college").mean()
    cluster_ids    = zs_rows["row_id"].values
    top_occ        = sample_raw.loc[sample_raw.index.isin(cluster_ids), "occupation"].value_counts().head(3).index.tolist()
    avg_age        = sample_raw.loc[sample_raw.index.isin(cluster_ids), "age"].mean()
    verdicts       = zs_rows["audit_verdict"].dropna()
    dominant       = verdicts.value_counts().idxmax() if len(verdicts) > 0 else "unaudited"
    cluster_stats.append({
        "cluster": c, "size": size,
        "zs_accuracy": round(zs_acc, 3), "fs_accuracy": round(fs_acc, 3),
        "college_rate": round(college_rate, 3), "avg_age": round(avg_age, 1),
        "dominant_verdict": dominant,
        "top_occupations": ", ".join(top_occ),
    })
    print(f"\n── Cluster {c} ─────────────────────────────────────────")
    print(f"  Size:               {size} rows")
    print(f"  Zero-shot accuracy: {zs_acc:.1%}")
    print(f"  Few-shot accuracy:  {fs_acc:.1%}")
    print(f"  College rate:       {college_rate:.1%}")
    print(f"  Avg age:            {avg_age:.1f}")
    print(f"  Dominant verdict:   {dominant}")
    print(f"  Top occupations:    {", ".join(top_occ)}")

cluster_df = pd.DataFrame(cluster_stats)
cluster_df.to_csv(f"{RESULTS_DIR}/week3_cluster_analysis.csv", index=False)
print("\nSaved: results/week3_cluster_analysis.csv")


## 15. Cluster Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.barplot(data=cluster_df, x="cluster", y="zs_accuracy", color="#378ADD", ax=axes[0])
axes[0].axhline(y=zs_df["correct"].mean(), color="red", linestyle="--",
                label=f'Overall ({zs_df["correct"].mean():.1%})')
axes[0].set_title("Zero-shot Accuracy by Cluster")
axes[0].set_ylim(0, 1)
axes[0].legend(fontsize=8)
for i, row in cluster_df.iterrows():
    axes[0].text(i, 0.02, row["dominant_verdict"], ha="center",
                 fontsize=8, color="white", fontweight="bold")

sns.barplot(data=cluster_df, x="cluster", y="fs_accuracy", color="#1D9E75", ax=axes[1])
axes[1].axhline(y=fs_df["correct"].mean(), color="red", linestyle="--",
                label=f'Overall ({fs_df["correct"].mean():.1%})')
axes[1].set_title("Few-shot Accuracy by Cluster")
axes[1].set_ylim(0, 1)
axes[1].legend(fontsize=8)

sns.barplot(data=cluster_df, x="cluster", y="college_rate", color="#EF9F27", ax=axes[2])
axes[2].set_title("College Rate by Cluster (true labels)")
axes[2].set_ylim(0, 1)

plt.suptitle(f"K-Means Cluster Analysis (k={K}) — Nano 4B", fontsize=12)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/week3_cluster_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: results/week3_cluster_accuracy.png")


## 16. Save Results to metrics.csv

In [ ]:
metrics_path = f"{RESULTS_DIR}/metrics.csv"
try:
    existing = pd.read_csv(metrics_path)
    existing = existing[existing["week"] != 3]
    updated  = pd.concat([existing, pd.DataFrame(all_results)], ignore_index=True)
except FileNotFoundError:
    updated = pd.DataFrame(all_results)

updated.to_csv(metrics_path, index=False)
print(f"Saved: {metrics_path}")
print()
print(updated[["model", "accuracy", "macro_f1", "auc_roc", "ms_per_row", "week"]].to_string(index=False))


## 17. Week 3 Summary

Fill in after running all cells:

| Model | Accuracy | Macro F1 | AUC-ROC | Time/row |
|---|---|---|---|---|
| Random Forest (Week 1) | 77.09% | 0.7701 | 0.8556 | 0.007 ms |
| XGBoost (Week 1) | 71.50% | 0.7149 | 0.7812 | 0.001 ms |
| Nano 4B zero-shot | ___ | ___ | ___ | ___ ms |
| Nano 4B few-shot | ___ | ___ | ___ | ___ ms |

**Accuracy by occupation difficulty:**

| Verdict | Rows | Zero-shot | Few-shot |
|---|---|---|---|
| Yes (clearly college) | ___ | ___ | ___ |
| No (clearly not college) | ___ | ___ | ___ |
| Maybe (ambiguous) | ___ | ___ | ___ |

**Commit before moving on:**
```
git add .
git commit -m "Week 3: occupation audit + experiments + accuracy by difficulty + K-Means"
git push
```
